### Taks 1.2 - Create the fact_transactions table with incremental update


In [0]:
# Perform incremental update with data from parqueut
# create a new parquet file with the new transaction data; then insert the new tranasaction into the fact table

from pyspark.sql.functions import lit, current_date, col, expr

catalog_name = dbutils.widgets.get("catalog_name")

# prepare a new transaction
df_tran_new = (
    spark.read.table(f"{catalog_name}.2_silver.fact_transactions")
    .orderBy("transaction_id", ascending=False)
    .limit(1)
    .withColumn(
        "transaction_id",
        expr(
            "CASE WHEN int(right(transaction_id, 3)) < 100 THEN 'TXN-100' "
            "ELSE concat('TXN-', lpad(cast(int(right(transaction_id, 3)) + 1 as string), 3, '0')) END"
        )
    )
    .withColumn("account_id", lit("ACC-400"))
    .withColumn("transaction_type", lit("Sales"))
    .withColumn("transaction_date", current_date())
    .withColumn("description", lit("Sales"))
)

# prepare an update transaction
df_tran_update = (
    spark.read.table(f"{catalog_name}.2_silver.fact_transactions")
    .where(col("transaction_id") == "TXN-001")
    .limit(1)
    .withColumn("amount", col("amount") + 1)
)
# put them together
df_tran_all = df_tran_new.unionByName(df_tran_update)



# write it a parquet file
df_tran_all.write.mode("overwrite").format("parquet").save(f"/Volumes/{catalog_name}/1_bronze/raw_files/NewTransactions.parquet")

# read from a parquet file
df_staging = spark.read.parquet(f"/Volumes/{catalog_name}/1_bronze/raw_files/NewTransactions.parquet")
df_staging.createOrReplaceTempView("staging_tran")

# Use MERGE to update existing transactions or insert new ones
spark.sql(f"""
MERGE INTO {catalog_name}.2_silver.fact_transactions AS target
USING staging_tran AS source
ON target.transaction_id = source.transaction_id
WHEN MATCHED THEN
  UPDATE SET
    target.account_id        = source.account_id,
    target.customer_id       = source.customer_id,
    target.transaction_type  = source.transaction_type,
    target.amount            = source.amount,
    target.currency          = source.currency,
    target.transaction_date  = source.transaction_date,
    target.description       = source.description
WHEN NOT MATCHED THEN
  INSERT (
    transaction_id,
    account_id,
    customer_id,
    transaction_type,
    amount,
    currency,
    transaction_date,
    description
  )
  VALUES (
    source.transaction_id,
    source.account_id,
    source.customer_id,
    source.transaction_type,
    source.amount,
    source.currency,
    source.transaction_date,
    source.description
  )
""")

In [0]:
%sql
    
-- verify incremental update on fact_transaction table
(SELECT  * FROM IDENTIFIER(:catalog_name || '.2_silver.fact_transactions') ORDER BY transaction_id ASC LIMIT 1)
UNION ALL
(SELECT * FROM IDENTIFIER(:catalog_name || '.2_silver.fact_transactions') ORDER BY transaction_id DESC LIMIT 1)